# DEMO DIVRS — Khử thiên lệch (ML-10M)

So sánh **DIVRS** với baseline **Most-Popular**.
**KHÔNG train, KHÔNG cần GPU** — chỉ nạp weight DIVRS đã có trên Drive.
Gồm: (A) bảng số (Recall/HR/NDCG + độ phổ biến), (B) đường cong, (C) **web demo** có đánh dấu ✅ phim user thật sự xem.

## 1. Setup: code + data + Drive

In [ ]:
!pip install -q faiss-cpu gradio matplotlib
import os, glob, re
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'

## 2. Nạp embedding DIVRS (checkpoint best trên Drive) + dữ liệu

In [ ]:
import numpy as np, scipy.sparse as sp, torch
D='data/ml10m/output/'
train=sp.load_npz(D+'train_coo_record.npz').tocsr()
test =sp.load_npz(D+'test_coo_record.npz').tocsr()
pop  =np.load(D+'popularity.npy').astype(float)
n_user,n_item=train.shape
pop_pct=pop.argsort().argsort()/(len(pop)-1)   # 0=it pho bien, 1=pho bien nhat

def best_ckpt(run):
    cks=glob.glob(run+'ckpt/epoch_*.pth')
    best_ep,best_rc=None,-1
    for lf in glob.glob(run+'log/*'):
        try: txt=open(lf,encoding='utf-8',errors='ignore').read()   # bo qua symlink .INFO hong tren Drive
        except Exception: continue
        for m in re.finditer(r"VALIDATION epoch: (\d+).*?'recall': np\.float64\(([\d.]+)\)", txt):
            ep,rc=int(m.group(1)),float(m.group(2))
            if rc>best_rc: best_rc,best_ep=rc,ep
    if best_ep is not None and os.path.exists(run+f'ckpt/epoch_{best_ep}.pth'):
        return run+f'ckpt/epoch_{best_ep}.pth', best_ep, round(best_rc,4)
    mx=max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))
    return mx, int(re.findall(r'epoch_(\d+)',mx)[0]), None

div_runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
assert div_runs, 'Khong thay run DIVRS nao co checkpoint tren Drive!'
div_run=max(div_runs, key=lambda r: len(glob.glob(r+'ckpt/epoch_*.pth')))
ckpt,ep,rc=best_ckpt(div_run)
sd=torch.load(ckpt,map_location='cpu')
Ud=sd['users_iv'].float().numpy(); Id=sd['items_iv'].float().numpy()
print('DIVRS run:',div_run)
print('Dung checkpoint: epoch',ep,'| val recall:',rc)
print('embedding:',Ud.shape,Id.shape,'| users',n_user,'items',n_item)

def score_divrs(u): return Ud[u] @ Id.T
def score_pop(u):   return pop
MODELS={'Most-Popular (baseline)':score_pop, 'DIVRS (debiased)':score_divrs}

## A. Bảng so sánh — đủ metric paper: Recall / HR / NDCG + Độ phổ biến (proxy IOU)

In [ ]:
def evaluate(score_fn,k=20,n_eval=2000):
    users=np.where(np.diff(test.indptr)>0)[0][:n_eval]
    rec,hr,ndcg,popm=[],[],[],[]
    idcg_cache={}
    for u in users:
        s=np.array(score_fn(u),dtype=float); s[train[u].indices]=-1e9
        top=np.argsort(-s)[:k]
        tset=set(test[u].indices)
        hits=[1 if it in tset else 0 for it in top]
        nh=sum(hits)
        rec.append(nh/max(len(tset),1))
        hr.append(1.0 if nh>0 else 0.0)
        dcg=sum(h/np.log2(i+2) for i,h in enumerate(hits))
        m=min(len(tset),k); idcg=idcg_cache.get(m) or sum(1/np.log2(i+2) for i in range(m)); idcg_cache[m]=idcg
        ndcg.append(dcg/idcg if idcg>0 else 0.0)
        popm.append(pop_pct[top].mean())
    return np.mean(rec),np.mean(hr),np.mean(ndcg),np.mean(popm)

print(f"{'Model':<26}{'Recall@20':>11}{'HR@20':>9}{'NDCG@20':>10}{'AvgPop%':>10}")
for name,fn in MODELS.items():
    r,h,n,p=evaluate(fn)
    print(f'{name:<26}{r:>11.4f}{h:>9.4f}{n:>10.4f}{p*100:>9.1f}%')
print('\n=> DIVRS: Recall/HR/NDCG cao hon, AvgPop% thap hon = chinh xac hon va it thien lech hon.')

## B. Đường cong: độ phổ biến TB của gợi ý theo Top-K (thấp = khử thiên lệch tốt)

In [ ]:
import matplotlib.pyplot as plt
Ks=[5,10,20,30,40,50]
plt.figure(figsize=(6,4))
for name,fn in MODELS.items():
    y=[evaluate(fn,k=k,n_eval=1000)[3]*100 for k in Ks]
    plt.plot(Ks,y,marker='o',label=name)
plt.xlabel('Top-K'); plt.ylabel('Do pho bien TB cua goi y (%)')
plt.title('Khu thien lech: DIVRS thap hon baseline'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('debias_curve.png',dpi=120,bbox_inches='tight'); plt.show()
print('Da luu debias_curve.png')

## C. WEB DEMO — ✅ = phim user THẬT SỰ xem (tập test giấu đi)
Kéo User ID (thử các số được in ra). `live=True` tự cập nhật. DIVRS trúng nhiều ✅ hơn = gợi ý đúng hơn.

In [ ]:
import gradio as gr
GOOD=np.argsort(-np.diff(test.indptr))[:30].tolist()   # user nhieu phim test -> de thay 'trung'
print('User ID nen thu (nhieu phim test):', GOOD[:12])

def reco(uid,k):
    uid=int(uid); k=int(k); watched=set(test[uid].indices); res=[]
    for name,fn in MODELS.items():
        s=np.array(fn(uid),dtype=float); s[train[uid].indices]=-1e9
        top=np.argsort(-s)[:k]
        lines=[]
        for r,it in enumerate(top):
            mark='   ✅ user CO xem that' if it in watched else ''
            lines.append(f'#{r+1:>2}  Item {it:<5}  pho bien:{pop_pct[it]*100:>4.0f}%{mark}')
        nh=len(set(top)&watched)
        res.append('\n'.join(lines)+f'\n\n>>> TRUNG {nh}/{k} phim user that su xem  |  pho bien TB: {pop_pct[top].mean()*100:.0f}%')
    return res[0],res[1]

gr.Interface(reco,
    [gr.Slider(0,n_user-1,step=1,value=int(GOOD[0]),label='User ID (thu cac so in o tren)'),
     gr.Slider(5,20,step=1,value=10,label='Top-K')],
    [gr.Textbox(label='Most-Popular baseline',lines=14),
     gr.Textbox(label='DIVRS (debiased)',lines=14)],
    title='DIVRS Debiasing Demo — ML-10M',
    description='✅ = phim user THUC SU xem (data giau di de kiem tra). DIVRS trung nhieu hon + it pho bien hon.',
    live=True
).launch(share=True)

## Ghi chú
- ✅ chỉ đánh dấu phim nằm trong **tập test (đã giấu)** — là **mẫu** các phim user xem, không phải toàn bộ. Nên **không-✅ ≠ user sẽ không xem**, chỉ là chưa kiểm chứng được.
- Số **chính thức cho báo cáo** lấy từ `test_log` của run train. Bảng demo eval trên 2000 user đầu nên lệch chút.
- `AvgPop%` = proxy IOU. Thấp hơn = khử thiên lệch tốt hơn.